# 1: PREPARAMOS NUESTRO ENTORNO

In [6]:
# Preparamos nuestro entorno para google colab.
%pip install -q -U openai google-genai langchain langchain-openai langchain-google-genai langgraph python-dotenv tiktoken pydantic
%pip install -q -U google-genai langchain langchain-core langchain-community langchain-google-genai langchain-text-splitters langgraph python-dotenv pypdf
%pip install -q -U openai openai-agents
%pip install -q -U openai python-dotenv numpy pandas requests==2.32.4 jsonschema

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


Generamos las variables de entorno. Vamos a trabajar directamente con mi clave de OPENAI, si que la meteremos directamente en el codigo

In [1]:
import time
import os, getpass
from dotenv import load_dotenv

import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Literal

import numpy as np
import pandas as pd
import requests
from jsonschema import Draft202012Validator
from openai import OpenAI


# Intenta cargar las variables desde el archivo .env
dotenv_loaded = load_dotenv()

# Solicita las claves si no están definidas
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


# Si después de todo no hay claves, lanza un error
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("No se encontraron las API keys requeridas. Por favor, agrégalas al archivo .env o introdúcelas manualmente.")

print("Claves cargadas en variables de entorno.")

Claves cargadas en variables de entorno.


# 2: CREAMOS EL CLIENTE DE OPENAI Y LO PROBAMOS


In [2]:
#Instanciamos el modelo de openAI.
# Elegimos el modelo 4.1 nano ya que este modelo es economico pero suficientemente potente como para realizar la tarea
from openai import OpenAI

OPENAI_MODEL = "gpt-4.1-nano"

# Instanciamos el cliente.
cliente = OpenAI() # Para OPEN AI

In [3]:
# Primera prueba con mi modelo.
prompt_saludo = "Saludame que es la primera vez que estamos solos. Espero que este sea el comienzo de una gran amistad. "

response = cliente.responses.create( # Metodo responses.create
    model=OPENAI_MODEL, # Le pasamos el modelo con los parametros:
    input=prompt_saludo, # Le pasamos el prompt
    max_output_tokens=80,
    temperature=0.8, # Para que sea creativo
    stream=True
)
# Vemos su respuesta:
#print(response.output_text)
# Probamos nuestro modelo en modo streaming
for event in response:
    if event.type == "response.output_text.delta":
        time.sleep(0.01)
        print(event.delta, end="")

¡Hola! Qué bonito mensaje, me alegra mucho poder conversar contigo. Espero que esta sea la primera de muchas charlas y que podamos construir una gran amistad. ¿En qué puedo ayudarte hoy?

# 3: RAG

## 3.1: Cargamos nuestro documento

In [ ]:
# Primero cargamos nuestro documento:
from langchain_community.document_loaders import PyPDFLoader # Louder para pdf de langchain
from pathlib import Path

# Para cargar el PDF en colab
#from google.colab import drive
#drive.mount('/content/drive')


PDF_PATH_COLAB = Path("/content/drive/MyDrive/PONTIA/08_Large Languaje Models/Entrega Final/data/TENERIFE.pdf") # Vamos a cargar este PDF
PDF_PATH_LOCAL = Path("data/TENERIFE.pdf") # Vamos a cargar este PDF

# Elegimos la opcion de Colab puesto que estoy trabajando en la nube.
PDF_PATH = PDF_PATH_LOCAL


if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

loader = PyPDFLoader(str(PDF_PATH))
docs = loader.load()

print("Número de páginas cargadas:", len(docs))
print("Metadata de la primera página:", docs[0].metadata)
print("Primeros caracteres:\n")
print(docs[0].page_content[:700])

C:\Users\Jesus\AppData\Local\Temp\ipykernel_19884\2275664481.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader # Louder para pdf de langchain


Número de páginas cargadas: 25
Metadata de la primera página: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': 'data\\TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1'}
Primeros caracteres:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - 
ubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del 
Príncipe - ubi

## 3.2: **Chunking**

In [ ]:
from langchain_openai import OpenAIEmbeddings # Importamos la clase correcta de Open AI para usar sus embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter # Importamos el text_splitter de texto

# 1: Cliente de embedding
# Voy a utilizar el embedding de openAI
EMBEDDING_MODEL = "text-embedding-ada-002"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL) # Creamos el cliente para los embeddings de OpenAI

# Configuramos nuestro troceador de documentos.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # tamaño del chunk en caracteres como maximo
    # Si llega a un separador como saltos de linea, ., etc. va a cortar. Va a buscar el ultimo separador disponible, si no corta en los 500
    # Cuanto mas grande mas general es la informacion.
    chunk_overlap=80, # Cuantos caracteres quiero de solapamiento entre un chunk y otro
    add_start_index=True,
)
# Aqui es donde troceamos nuestro texto con los parametros que hemos elegido
splits = text_splitter.split_documents(docs) # Docs es donde tenemos nuestro pdf.

# Añadimos un identificador simple a cada chunk para poder citarlo luego.
for i, doc in enumerate(splits):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(splits))
print("\nEjemplo de chunk:\n")
print(splits[0].page_content)
print("\nMetadata:", splits[0].metadata)
# Vemos el primer chunk y sus metadatos.

Número de chunks: 47

Ejemplo de chunk:

TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo

Metadata: {'producer': 'PyPDF', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'title': 'Microsoft Word - TENERIFE.docx', 'moddate': '2025-07-13T20:00:01+00:00', 'source': 'data\\TENERIFE.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


In [6]:
# Listamos todos los chunks que han salido
for doc in splits:
    print(f"Chunk {doc.metadata['chunk_id']} | Página {doc.metadata.get('page')} | Caracteres: {len(doc.page_content)}")

Chunk 0 | Página 0 | Caracteres: 486
Chunk 1 | Página 0 | Caracteres: 472
Chunk 2 | Página 1 | Caracteres: 86
Chunk 3 | Página 2 | Caracteres: 439
Chunk 4 | Página 2 | Caracteres: 158
Chunk 5 | Página 3 | Caracteres: 379
Chunk 6 | Página 4 | Caracteres: 422
Chunk 7 | Página 4 | Caracteres: 431
Chunk 8 | Página 5 | Caracteres: 124
Chunk 9 | Página 7 | Caracteres: 455
Chunk 10 | Página 8 | Caracteres: 490
Chunk 11 | Página 8 | Caracteres: 495
Chunk 12 | Página 8 | Caracteres: 253
Chunk 13 | Página 9 | Caracteres: 219
Chunk 14 | Página 10 | Caracteres: 486
Chunk 15 | Página 10 | Caracteres: 430
Chunk 16 | Página 10 | Caracteres: 394
Chunk 17 | Página 11 | Caracteres: 488
Chunk 18 | Página 11 | Caracteres: 469
Chunk 19 | Página 11 | Caracteres: 458
Chunk 20 | Página 11 | Caracteres: 443
Chunk 21 | Página 12 | Caracteres: 162
Chunk 22 | Página 13 | Caracteres: 373
Chunk 23 | Página 14 | Caracteres: 474
Chunk 24 | Página 14 | Caracteres: 493
Chunk 25 | Página 14 | Caracteres: 22
Chunk 26 | P

## 3.3: Vector Store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
# Vamos a guardar los vectores en memoria. En la variable documents_ids y en vector_store
vector_store = InMemoryVectorStore(embeddings) # Usamos nuestro embeddings para transformar a vectores.
document_ids = vector_store.add_documents(splits) # Añadimos los chuncks y los convierte en su respesentacion vectorial

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 47
Primeros IDs: ['f3832f1c-21b5-46e6-bd90-f6cbe05d248c', 'd2ba3f20-9894-48a3-9597-5109c94e4d96', 'c61c5ced-2a04-4617-9039-1d2b81419d25']


## 3.4: Funcion que nos devuelva el retrieve

In [39]:
# Importamos las librerias necesarias:
from agents import Agent, Runner, function_tool, ModelSettings, ToolCallItem, ToolCallOutputItem

# Funcion para mostrar los documentos recuperados
def mostrar_documentos_recuperados(documentos):
    for i, doc in enumerate(documentos, start=1):
        print(f"Resultado {i}")
        print("Fuente:", doc.metadata.get("source_name"))
        print("Página:", doc.metadata.get("page"))
        print("Chunk:", doc.metadata.get("chunk_id"))
        print(doc.page_content)
        print("-" * 30)

# Ahora con el decorador @function_tool creamos la herramienta para nuestro agente que devolvera el contexto donde tiene que buscar.

@function_tool
def rag_retrieve(query: str): # Recibe una query del agente.
    """
    Busca información de turismo de la Isla de Tenerife.
    """
    results = vector_store.similarity_search(query, k=5) # Buscamos dentro de nuestra vector data base los chuncks mas relevantes

    documentos = [
        {
            "content": doc.page_content,
            "metadata": doc.metadata
        }
        for doc in results
    ]

    # Mostrar por pantalla los documentos recuperados
    mostrar_documentos_recuperados(results)

    return documentos
    


# 4: API EXTERNA PARA CONSULTAR EL TIEMPO

In [41]:
from typing import Any, Callable, Literal
import requests

# Funcion qeu nos devuelve la latitud, longitud, country, hora y nombre.
def geocode_location(location: str) -> dict[str, Any]:
    """Obtiene coordenadas aproximadas para una ubicación usando Open-Meteo Geocoding."""
    response = requests.get( ## Llamamos a la api para que nos de la geolocalizacion.
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "es", "format": "json"},
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results") or []
    if not results:
        raise ValueError(f"No se encontraron coordenadas para: {location}")

    first = results[0]
    return {
        "name": first.get("name"),
        "country": first.get("country"),
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "timezone": first.get("timezone"),
    }

# Esta es la funcion que le vamos a dar como herramienta al LLM
# Le vamos a pedir la ubicacion y las unidades en celsius o Fahrenheit
@function_tool
def get_current_weather(location: str, units: Literal["celsius", "fahrenheit"], date: str) -> dict[str, Any]:
    """Consulta el tiempo para una ubicación y una fecha específica.
       La fecha debe estar en formato YYYY-MM-DD

    Args:
        location (str): Ciudad o ubicación. Ejemplos: Madrid, Illescas, Bogotá, Paris.
        units (Literal["celsius", "fahrenheit"]): Unidad de temperatura deseada por el usuario.
        date:str: Fecha de la prediccion del tiempo
    """
    place = geocode_location(location)
    temperature_unit = "fahrenheit" if units == "fahrenheit" else "celsius"

    response = requests.get( # Llamamos a la api del tiempo.
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
          #  "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
            "temperature_unit": temperature_unit,
            "timezone": "auto",
            "start_date": date,
            "end_date": date,
        },
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()

    #current = payload.get("current") or {}
    #units_payload = payload.get("current_units") or {}

    # Usamso daily porque es el end point que permite meter fechas
    daily = payload.get("daily") or {}
    units_payload = payload.get("daily_units") or {}
    # Todo esto es lo que devuelve la funcion.

    return {
        # Esto seria actualmente usando current

       # "location": place,
       # "observed_at": current.get("time"),
       # "temperature": current.get("temperature_2m"),
       # "temperature_unit": units_payload.get("temperature_2m"),
       # "relative_humidity": current.get("relative_humidity_2m"),
       # "relative_humidity_unit": units_payload.get("relative_humidity_2m"),
       # "wind_speed": current.get("wind_speed_10m"),
       # "wind_speed_unit": units_payload.get("wind_speed_10m"),
       # "provider": "Open-Meteo",

        # Esto para una fecha usando daily
        "location": place,
        "date": date,
        "temperature_max": daily.get("temperature_2m_max", [None])[0],
        "temperature_max_unit": units_payload.get("temperature_2m_max"),
        "temperature_min": daily.get("temperature_2m_min", [None])[0],
        "temperature_min_unit": units_payload.get("temperature_2m_min"),
        "precipitation_sum": daily.get("precipitation_sum", [None])[0],
        "precipitation_unit": units_payload.get("precipitation_sum"),
        "provider": "Open-Meteo",
    }


In [97]:
#Probamos nuestra funcion: Quitar el @FunctionTool para probarla
#get_current_weather("Sonseca", "celsius", "2026-07-21" )

{'location': {'name': 'Sonseca',
  'country': 'España',
  'latitude': 39.67747,
  'longitude': -3.97448,
  'timezone': 'Europe/Madrid'},
 'date': '2026-07-21',
 'temperature_max': 41.5,
 'temperature_max_unit': '°C',
 'temperature_min': 27.1,
 'temperature_min_unit': '°C',
 'precipitation_sum': 0.0,
 'precipitation_unit': 'mm',
 'provider': 'Open-Meteo'}

# 5: AGENTE Con herramientas RAG y Weather

Aqui definimos los parametros de nuestro LLM de OPENAI

In [42]:
# Parametros de nuestro LLM
settings = ModelSettings(
    temperature=0.1, # Para que sea lo mas preciso posible
    top_p=0.1,
    max_output_tokens=600,
    stream=True
)

Ahora creamos el agente, y le metemos la herramientas que hemos creado


In [43]:
# Agente. Le paso el modelo LLM que ya teniamos definido, las instrucciones y las herramientas que he creado
tenerife_agent = Agent(
    name="AgenteTenerife",
    model=OPENAI_MODEL,
    model_settings= settings,
    instructions=(
        "Eres un guia turistico de Tenerife. "
        "Tu trabajo es responder sobre temas relacionados sobre el turismo en Tenerife."
        "Si el contexto no contiene la respuesta, di no tengo informacion al respecto y sugiere consultar otras fuentes. "
        "Incluye siempre las fuentes usadas cuando respondas a partir de documentación. "
        "Responde en español, con precisión y sin inventar detalles externos."
        "NO respondas preguntas sobre Tenerife sin consultar la herramienta rag_retrieve"
        "Si preguntan por el tiempo, usa la herramienta get_current_weather y no añadas mas informacion que la obtenida de la herramienta "
    ),
     tools=[rag_retrieve, get_current_weather]
)


## 5.1: Creamos la conversacion

Creamos lista de historial de mensajes y funciones para añadir los mensajes de usuario y de asistente:

In [44]:
# Multiturno, Creamos variable de conversaciones y dos metodos para añadir al historial
historial = []

def add_user(msg):
    historial.append({"role": "user", "content": msg})

def add_assistant(msg):
    historial.append({"role": "assistant", "content": msg})
print(historial)

[]


### Generamos el turno 1:

In [45]:
# Turno 1.
# Metemos la pregunta a la lista de preguntas de usuario
add_user("Creame un itinerario para visitar Santa Cruz de tenerife. Por cierto me llamo Jesus")

# Llamamos al agente. Le metemos el historial actualizado
respuesta_1 = await Runner.run(tenerife_agent, historial)

# Metemos la respuesta en la lista de respuestas del asistente
add_assistant(respuesta_1.final_output)

# Imprimimos el resultado
print("=" *50)
print(respuesta_1.final_output)
print("=" *50)

# Vemos datos de la respuesta:
print("Último agente:", respuesta_1.last_agent.name)
print("Tipo de salida final:", type(respuesta_1.final_output).__name__)
print("Número de items nuevos:", len(respuesta_1.new_items))
print("Tipos de items generados:")
for item in respuesta_1.new_items:
    print("-", type(item).__name__)
    if isinstance(item, ToolCallItem):
        print(f" -- Se llamó a la herramienta: {item.tool_name}")


Resultado 1
Fuente: TENERIFE.pdf
Página: 0
Chunk: 0
TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo
------------------------------
Resultado 2
Fuente: TENERIFE.pdf
Página: 2
Chunk: 3
o Parque García Sanabria [vídeo - ubicación] 
o Playa de Las Teresitas [vídeo - ubicación] 
 
 
 
 
• San Cristóbal de La Laguna 
Es ciudad Patrimonio de la Humanidad. Merece la pena un paseo por el centro, 
repleto de edificios de estilo colonial. Quizás la ruta a seguir sería: 
o Aparcar cerca de la Plaza de la Concepción [ubicación], donde se encuentra la 
Iglesia de la Co

### Generamos el turno 2:

In [37]:
# Turno 2.
# Metemos la pregunta a la lista de preguntas de usuario
add_user(" ¿Como me llamo?.")

# Llamamos al agente. Le metemos el historial actualizado
respuesta_2 = await Runner.run(tenerife_agent, historial)

# Metemos la respuesta en la lista de respuestas del asistente
add_assistant(respuesta_2.final_output)

# Imprimimos el resultado
print("=" *50)
print(respuesta_2.final_output)
print("=" *50)

# Vemos datos de la respuesta:
print("Último agente:", respuesta_2.last_agent.name)
print("Tipo de salida final:", type(respuesta_2.final_output).__name__)
print("Número de items nuevos:", len(respuesta_2.new_items))
print("Tipos de items generados:")
for item in respuesta_2.new_items:
    print("-", type(item).__name__)
    if isinstance(item, ToolCallItem):
        print(f" -- Se llamó a la herramienta: {item.tool_name}")


Te llamas Jesús.
Último agente: AgenteTenerife
Tipo de salida final: str
Número de items nuevos: 1
Tipos de items generados:
- MessageOutputItem


### Generamos el turno 3:

In [46]:
# Turno 3.
# Metemos la pregunta a la lista de preguntas de usuario
add_user("¿Cual era el punto 1 que me has recomendado en la primera pregunta? y que tiempo hace el 2026-07-16?.")
# Llamamos al agente. Le metemos el historial actualizado
respuesta_3 = await Runner.run(tenerife_agent, historial)

# Metemos la respuesta en la lista de respuestas del asistente
add_assistant(respuesta_3.final_output)

# Imprimimos el resultado
print("=" *50)
print(respuesta_3.final_output)
print("=" *50)

# Vemos datos de la respuesta:
print("Último agente:", respuesta_3.last_agent.name)
print("Tipo de salida final:", type(respuesta_3.final_output).__name__)
print("Número de items nuevos:", len(respuesta_3.new_items))
print("Tipos de items generados:")
for item in respuesta_3.new_items:
    print("-", type(item).__name__)
    if isinstance(item, ToolCallItem):
        print(f" -- Se llamó a la herramienta: {item.tool_name}")



Resultado 1
Fuente: TENERIFE.pdf
Página: 0
Chunk: 0
TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo
------------------------------
Resultado 2
Fuente: TENERIFE.pdf
Página: 10
Chunk: 16
Charco (los helados del Freddino no están mal por si os apetece) y, de ahí, de vuelta 
al coche. 
En el Puerto de la Cruz También se encuentra el Loro Parque, que ha sido 
considerado varios años como el Mejor Zoológico del Mundo. Normalmente es un 
must que hay que ver en las turistadas de Tenerife (entradas aquí) y, cerca, tenéis el 
Brunelli's Steakhouse por si os queréi